In [2]:
#!/usr/bin/env python3

"""
Generate kcat-ablation ecGEMs from final EMMAi ecGEMs.

Conditions
----------
1. original EMMAi
2. uniform median reaction-level kcat
3. 50 shuffled reaction-level kcat assignments

Only reactions for which the original reaction-level kcat can be reconstructed
are altered. Reactions that received EMMAi's fallback/missing-cost treatment
are left unchanged.

Everything else in the final ecGEM remains unchanged:
- reaction structure
- reaction bounds / exchanges
- objective
- protein-pool calibration
- forward/reverse splitting
- isozyme splitting
- missing-cost handling

Outputs per model
-----------------
<model>/
    uniform_median.xml
    shuffle_001.xml
    ...
    shuffle_050.xml
    kcat_ablation_audit.csv
    kcat_ablation_summary.csv
"""

from pathlib import Path

import cobra
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------

SUPPLEMENTARY_FILE_3 = Path("C:/Users/mol131/OneDrive - CSIRO/ecGEMs_Paper/Submission_files/Supplementary file 3.xlsx")
EC_MODEL_PATH = "C:/Users/mol131/OneDrive - CSIRO/ecGEMs_Paper/Submission_files/Supplementary file 1/"
OUTPUT_ROOT = Path("kcat_ablation_models")

N_SHUFFLES = 50
RANDOM_SEED = 20260925


MODELS = {
    "eciML1515": {
        "model_path": Path(f"{EC_MODEL_PATH}/iML1515/eciML1515.xml"),
        "sequence_sheet": "4. Sequences_SMILES_eciML1515",
    },
    "eciJO1366": {
        "model_path": Path(f"{EC_MODEL_PATH}/iJO1366/eciJO1366.xml"),
        "sequence_sheet": "7. Sequence_SMILES_iJO1366",
    },
    "eciMO1056": {
        "model_path": Path(f"{EC_MODEL_PATH}/iMO1056/eciMO1056.xml"),
        "sequence_sheet": "10. Sequence_SMILES_iMO1056",
    },
    # "ecPAO1": {
    #     "model_path": Path(f"{EC_MODEL_PATH}/PAO1/ecPAO1.xml"),
    #     "sequence_sheet": "13. Sequence_SMILES_PAO1",
    # },
    # "ecBALT": {
    #     "model_path": Path(f"{EC_MODEL_PATH}/BALT/ec_BALT_gapfilled.xml"),
    #     "sequence_sheet": "16. Sequence_SMILES_BALT",
    # },
    # "ecPRET": {
    #     "model_path": Path(f"{EC_MODEL_PATH}/PRET/ec_PRET_gapfilled.xml"),
    #     "sequence_sheet": "19. Sequence_SMILES_PRET",
    # },
}


# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def get_usage_metabolite(model):
    """
    Find EMMAi's usage pseudo-metabolite.
    """
    for metabolite_id in ("usage", "M_usage"):
        if metabolite_id in model.metabolites:
            return model.metabolites.get_by_id(metabolite_id)

    raise KeyError("Could not find usage pseudo-metabolite.")


def get_original_coefficient(reaction, usage):
    """
    Return the usage coefficient for a reaction, or None if unconstrained.
    """
    if usage not in reaction.metabolites:
        return None

    return float(reaction.metabolites[usage])


def effective_reaction_kcat(reaction, seq_df, usage_id):
    """
    Reconstruct the reaction-level kcat selected by EMMAi.

    EMMAi:
      1. collects kcats for valid enzyme/substrate pairs;
      2. averages values across genes for each substrate;
      3. takes the minimum substrate-specific mean.

    Returned kcat is in the original UniKP units (s^-1).
    """

    if reaction.id == "DM_usage":
        return None

    # Determine reaction direction exactly as EMMAi does.
    if reaction.lower_bound >= 0 and reaction.upper_bound > 0:
        direction = "Forward"
        metabolites = reaction.reactants

    elif reaction.lower_bound < 0 and reaction.upper_bound <= 0:
        direction = "Reverse"
        metabolites = reaction.products

    else:
        return None

    # usage was added after the original substrate selection,
    # so exclude it when reconstructing the reactant set.
    substrate_ids = {
        metabolite.id
        for metabolite in metabolites
        if metabolite.id != usage_id
    }

    preliminary_kcats = {}

    for gene in reaction.genes:

        subset = seq_df[
            seq_df["Gene ID normalized"] == gene.id
        ]

        if subset.empty:
            continue

        for _, row in subset.iterrows():

            kcat = row["Kcat"]

            if pd.isna(kcat):
                continue

            if row["Substrate ID"] not in substrate_ids:
                continue

            if str(row["Reaction ID"]) not in reaction.id:
                continue

            if row["Direction"] != direction:
                continue

            preliminary_kcats.setdefault(
                row["Substrate ID"], []
            ).append(float(kcat))

    if not preliminary_kcats:
        return None

    substrate_kcats = [
        np.mean(values)
        for values in preliminary_kcats.values()
    ]

    return float(min(substrate_kcats))


def replace_usage_coefficient(model, reaction_id, new_coefficient):
    """
    Replace the existing usage coefficient without touching anything else.
    """

    usage = get_usage_metabolite(model)
    reaction = model.reactions.get_by_id(reaction_id)

    reaction.add_metabolites(
        {usage: float(new_coefficient)},
        combine=False,
    )


def apply_kcat_assignment(
    model,
    reaction_table,
    assigned_kcats,
):
    """
    Apply a kcat assignment directly to the loaded model.

    Returns the new usage coefficients so they can be audited.
    """

    usage = get_usage_metabolite(model)

    new_coefficients = {}

    for reaction_id, new_kcat in assigned_kcats.items():

        row = reaction_table.loc[reaction_id]

        old_kcat = row["original_kcat"]
        old_coefficient = row["original_coefficient"]

        new_coefficient = (
            old_coefficient
            * old_kcat
            / new_kcat
        )

        reaction = model.reactions.get_by_id(reaction_id)

        reaction.add_metabolites(
            {usage: float(new_coefficient)},
            combine=False,
        )

        new_coefficients[reaction_id] = new_coefficient

    return new_coefficients


def restore_original_coefficients(
    model,
    reaction_table,
):
    """
    Restore the original EMMAi usage coefficients.
    """

    usage = get_usage_metabolite(model)

    for reaction_id, row in reaction_table.iterrows():

        reaction = model.reactions.get_by_id(reaction_id)

        reaction.add_metabolites(
            {usage: float(row["original_coefficient"])},
            combine=False,
        )


# ---------------------------------------------------------------------
# ONE MODEL
# ---------------------------------------------------------------------

def process_model(model_name, config, seed):

    print(f"\n--- {model_name} ---")

    model_path = config["model_path"]
    sequence_sheet = config["sequence_sheet"]

    output_dir = OUTPUT_ROOT / model_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------------
    # Load final calibrated ecGEM
    # --------------------------------------------------------------

    model = cobra.io.read_sbml_model(model_path)

    usage = get_usage_metabolite(model)

    # --------------------------------------------------------------
    # Load original EMMAi enzyme/substrate/kcat table
    # --------------------------------------------------------------

    seq_df = pd.read_excel(
        SUPPLEMENTARY_FILE_3,
        sheet_name=sequence_sheet,
    )

    seq_df["Kcat"] = pd.to_numeric(
        seq_df["Kcat"],
        errors="coerce",
    )

    seq_df["Gene ID normalized"] = (
        seq_df["Gene ID"]
        .astype(str)
        .str.replace(".", "_", regex=False)
    )

    # --------------------------------------------------------------
    # Recover reaction-level kcats used by EMMAi
    # --------------------------------------------------------------

    rows = []

    total_usage_reactions = 0
    n_with_genes = 0
    n_with_kcat = 0

    for reaction in model.reactions:

        coefficient = get_original_coefficient(
            reaction,
            usage,
        )

        if coefficient is None:
            continue

        if reaction.id == "DM_usage":
            continue

        total_usage_reactions += 1
        if len(reaction.genes) > 0:
            n_with_genes += 1

        kcat = effective_reaction_kcat(
            reaction,
            seq_df,
            usage.id,
        )
        if kcat is not None and kcat > 0:
            n_with_kcat += 1

        # No directly reconstructable kcat:
        # leave EMMAi's existing fallback coefficient unchanged.
        if kcat is None or kcat <= 0:
            continue

        rows.append(
            {
                "reaction_id": reaction.id,
                "original_kcat": kcat,
                "original_coefficient": coefficient,
            }
        )

    print(f"Usage-constrained reactions: {total_usage_reactions}")
    print(f"Usage reactions with genes: {n_with_genes}")
    print(f"Reactions with reconstructed kcat: {n_with_kcat}")

    if not rows:
        print("\nExample usage-constrained reactions:")
        shown = 0

        for reaction in model.reactions:
            if usage not in reaction.metabolites:
                continue
            if reaction.id == "DM_usage":
                continue

            print(
                reaction.id,
                "genes=", [g.id for g in reaction.genes],
                "bounds=", reaction.bounds,
            )

            shown += 1
            if shown == 10:
                break

        raise RuntimeError(
            f"No reaction-level kcats could be reconstructed for {model_name}. "
            "The final ecGEM reaction/gene IDs are not matching the Supplementary "
            "file 3 sequence/SMILES table as expected."
        )

    reaction_table = pd.DataFrame(
        rows,
        columns=[
            "reaction_id",
            "original_kcat",
            "original_coefficient",
        ],
    )

    if reaction_table.empty:
        raise RuntimeError(
            f"No reaction-level kcats reconstructed for {model_name}."
        )

    reaction_table = (
        reaction_table
        .set_index("reaction_id")
        .sort_index()
    )

    original_kcats = reaction_table[
        "original_kcat"
    ].to_numpy()

    median_kcat = float(
        np.median(original_kcats)
    )

    print(
        f"Usage-constrained reactions: "
        f"{total_usage_reactions}"
    )

    print(
        f"Direct kcat reactions: "
        f"{len(reaction_table)}"
    )

    print(
        f"Unchanged fallback reactions: "
        f"{total_usage_reactions - len(reaction_table)}"
    )

    print(
        f"Median reaction kcat: "
        f"{median_kcat:.6g}"
    )

    # --------------------------------------------------------------
    # Audit tables
    # --------------------------------------------------------------

    audit_rows = []
    summary_rows = []

    # Original EMMAi
    for reaction_id, row in reaction_table.iterrows():

        audit_rows.append(
            {
                "condition": "original",
                "replicate": 0,
                "reaction_id": reaction_id,
                "original_kcat": row["original_kcat"],
                "assigned_kcat": row["original_kcat"],
                "original_coefficient":
                    row["original_coefficient"],
                "new_coefficient":
                    row["original_coefficient"],
            }
        )

    summary_rows.append(
        summarise_condition(
            "original",
            0,
            original_kcats,
            reaction_table["original_coefficient"].to_numpy(),
            original_kcats,
        )
    )

    # --------------------------------------------------------------
    # Uniform median model
    # --------------------------------------------------------------

    uniform_assignment = {
        reaction_id: median_kcat
        for reaction_id in reaction_table.index
    }

    uniform_coefficients = apply_kcat_assignment(
        model,
        reaction_table,
        uniform_assignment,
    )

    cobra.io.write_sbml_model(
        model,
        output_dir / "uniform_median.xml",
    )

    for reaction_id, row in reaction_table.iterrows():

        audit_rows.append(
            {
                "condition": "uniform_median",
                "replicate": 0,
                "reaction_id": reaction_id,
                "original_kcat":
                    row["original_kcat"],
                "assigned_kcat":
                    median_kcat,
                "original_coefficient":
                    row["original_coefficient"],
                "new_coefficient":
                    uniform_coefficients[reaction_id],
            }
        )

    summary_rows.append(
        summarise_condition(
            "uniform_median",
            0,
            np.full(
                len(original_kcats),
                median_kcat,
            ),
            np.array(
                list(uniform_coefficients.values())
            ),
            original_kcats,
        )
    )

    restore_original_coefficients(
        model,
        reaction_table,
    )

    # --------------------------------------------------------------
    # Shuffled models
    # --------------------------------------------------------------

    rng = np.random.default_rng(seed)

    reaction_ids = reaction_table.index.to_list()

    for replicate in range(1, N_SHUFFLES + 1):

        shuffled_kcats = rng.permutation(
            original_kcats
        )

        # Exact check: shuffle must contain the same kcat values.
        if not np.array_equal(
            np.sort(shuffled_kcats),
            np.sort(original_kcats),
        ):
            raise RuntimeError(
                f"{model_name} shuffle {replicate}: "
                "kcat distribution changed."
            )

        assignment = dict(
            zip(
                reaction_ids,
                shuffled_kcats,
            )
        )

        shuffled_coefficients = apply_kcat_assignment(
        model,
        reaction_table,
        assignment,
    )

        filename = (
            f"shuffle_{replicate:03d}.xml"
        )

        cobra.io.write_sbml_model(
            model,
            output_dir / filename,
        )

        for reaction_id, new_kcat in assignment.items():

            row = reaction_table.loc[reaction_id]

            audit_rows.append(
                {
                    "condition": "shuffle",
                    "replicate": replicate,
                    "reaction_id": reaction_id,
                    "original_kcat":
                        row["original_kcat"],
                    "assigned_kcat":
                        new_kcat,
                    "original_coefficient":
                        row["original_coefficient"],
                    "new_coefficient":
                        shuffled_coefficients[
                            reaction_id
                        ],
                }
            )

        summary_rows.append(
            summarise_condition(
                "shuffle",
                replicate,
                shuffled_kcats,
                np.array(
                    list(
                        shuffled_coefficients.values()
                    )
                ),
                original_kcats,
            )
        )

        restore_original_coefficients(
            model,
            reaction_table,
        )

    # --------------------------------------------------------------
    # Save audit information
    # --------------------------------------------------------------

    audit_df = pd.DataFrame(audit_rows)

    audit_df.to_csv(
        output_dir / "kcat_ablation_audit.csv",
        index=False,
    )

    summary_df = pd.DataFrame(summary_rows)

    summary_df[
        "total_usage_reactions"
    ] = total_usage_reactions

    summary_df[
        "direct_kcat_reactions"
    ] = len(reaction_table)

    summary_df[
        "unchanged_fallback_reactions"
    ] = (
        total_usage_reactions
        - len(reaction_table)
    )

    summary_df.to_csv(
        output_dir / "kcat_ablation_summary.csv",
        index=False,
    )


def summarise_condition(
    condition,
    replicate,
    kcats,
    coefficients,
    original_kcats,
):

    same_kcat_distribution = np.array_equal(
        np.sort(kcats),
        np.sort(original_kcats),
    )

    return {
        "condition": condition,
        "replicate": replicate,

        "n": len(kcats),

        "kcat_min": np.min(kcats),
        "kcat_median": np.median(kcats),
        "kcat_mean": np.mean(kcats),
        "kcat_max": np.max(kcats),
        "kcat_std": np.std(kcats),

        "coefficient_abs_min":
            np.min(np.abs(coefficients)),
        "coefficient_abs_median":
            np.median(np.abs(coefficients)),
        "coefficient_abs_mean":
            np.mean(np.abs(coefficients)),
        "coefficient_abs_max":
            np.max(np.abs(coefficients)),
        "coefficient_abs_std":
            np.std(np.abs(coefficients)),

        "same_kcat_distribution_as_original":
            same_kcat_distribution,
    }


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------

def main():

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    for model_number, (
        model_name,
        config,
    ) in enumerate(MODELS.items()):

        process_model(
            model_name,
            config,
            seed=RANDOM_SEED + model_number,
        )


if __name__ == "__main__":
    main()


--- eciML1515 ---
Usage-constrained reactions: 2523
Usage reactions with genes: 2455
Reactions with reconstructed kcat: 1691
Usage-constrained reactions: 2523
Direct kcat reactions: 1691
Unchanged fallback reactions: 832
Median reaction kcat: 4.03291

--- eciJO1366 ---
Usage-constrained reactions: 2251
Usage reactions with genes: 2162
Reactions with reconstructed kcat: 1571
Usage-constrained reactions: 2251
Direct kcat reactions: 1571
Unchanged fallback reactions: 680
Median reaction kcat: 4.01817

--- eciMO1056 ---
Usage-constrained reactions: 1319
Usage reactions with genes: 1282
Reactions with reconstructed kcat: 1131
Usage-constrained reactions: 1319
Direct kcat reactions: 1131
Unchanged fallback reactions: 188
Median reaction kcat: 3.69997
